In [ ]:

import re
import zipfile
import shutil
import urllib.request
import os.path,glob

#ダウンロードしたいURLを入力する
URL = 'https://www.aozora.gr.jp/cards/000035/files/301_ruby_5915.zip'

title = ''
artist = ''

def get_title_artist(text):
    title = text.splitlines()[0]
    artist = text.splitlines()[1]
    return title, artist

def convert(download_text):
    binarydata = open(download_text, 'rb').read()
    text = binarydata.decode('shift_jis')
    title, artist = get_title_artist(text)
    # ルビ、注釈などの除去
    text = re.split(r'\-{5,}', text)[2]
    text = re.split(r'底本：', text)[0]
    text = re.sub(r'《.+?》', '', text)
    text = re.sub(r'［＃.+?］', '', text)
    text = text.strip()
    return title, artist, text

def download(url):
 # データファイルをダウンロードする
 zip_file = re.split(r'/', url)[-1]

 if not os.path.exists(zip_file):
     print('Download URL')
     print('URL:',url)
     urllib.request.urlretrieve(url, zip_file)
 else:
     print('Download File exists')

 # フォルダの生成
 dir, ext = os.path.splitext(zip_file)
 if not os.path.exists(dir):
     os.makedirs(dir)

 # zipファイルの展開
 zip_obj = zipfile.ZipFile(zip_file, 'r')
 zip_obj.extractall(dir)
 zip_obj.close()

 # zipファイルの削除
 os.remove(zip_file)

 # テキストファイルの抽出
 path = os.path.join(dir,'*.txt')
 list = glob.glob(path)
 return list[0]

download_text = download(URL)
title, artist, text = convert(download_text)

shutil.rmtree(os.path.dirname(download_text))


In [244]:
stopwords = ["とこ","所","なか","前","時","こと","所謂","あと","事","うち","そう","もん","方","ところ","やつ","とき","こんど","まま","ほう","ど","ほとんど","つもり","もの",'する', 'ある', 'いる', 'なる', 'こと', 'これ', 'それ', 'あれ', 'ため', 'よう', 'さん', 'くん', 'ちゃん']

In [ ]:
import collections
import MeCab
dictionary_path = os.path.join(os.getcwd(), f"{artist}.csv")
mecab = MeCab.Tagger("")

# 簡単な日本語の文を分析
result = mecab.parse(text)
node = mecab.parseToNode(text)

word_list = []
counter_tags = {"助数詞", "助数詞可能"}
counter_words = {"箱", "部屋", "学年", "例", "粒", "晩", "つき"}
while node:
    features = node.feature.split(',')
    if node.surface and features[0] == "名詞":
        word = node.surface
        is_number = features[1] in {"数", "数詞"}
        is_counter = False
        next_node = node.next
        if (
            is_number
            and next_node
            and next_node.surface
        ):
            next_features = next_node.feature.split(',')
            is_counter = (
                any(feature in counter_tags for feature in next_features[1:])
                or next_node.surface in counter_words
            )
            if is_counter:
                word += next_node.surface
                node = next_node

        if not is_number or is_counter:
            word_list.append(word)
    node = node.next
word_chain = ' '.join(word_list)
word_list = [word for word in word_list if word not in stopwords]

c=collections.Counter(word_list)
print(c.most_common(10))  # 出現頻度の高い順に10件表示

In [ ]:
from matplotlib.font_manager import FontProperties
from wordcloud import WordCloud
import matplotlib.pyplot as plt

fpath = r"c:\Windows\Fonts\MEIRYO.TTC"  # フォントファイルのパスを指定
wordcloud = WordCloud(width=800, height=400, font_path=fpath, collocations=False).generate(" ".join(word_list))
print(title, artist)

plt.figure(figsize=(18,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title(
    f"{title} / {artist}",
    color="black",
    fontproperties=FontProperties(fname=fpath, size=30)
)
plt.axis("off")
plt.show()